# Day 2 · Session 2 — Ultrasound tissue tracking (DUSTrack)

**MIT Professional Education · Mastering Integrated Systems**

In Session 1 you pulled information out of ordinary **video** — a network found the body's
joints in every frame. Now you do the same for **ultrasound**: follow points on the *tissue
inside* the forearm and turn that motion into a measurement.

Yesterday you *watched* the forearm tissue move and saw it line up with the muscle and the hand.
Today two tissue points were **tracked** through every frame with **DUSTrack** (a DeepLabCut
model cleaned up by an LK-RSTC filter). This notebook turns that tracking into a **number** — and
then asks you to design your own.

> Run-all works: `Runtime → Run all`. Same synchronized bundle as Day 1.

## Setup — same bundle as Day 1

In [ ]:
#@title Setup { display-mode: "form" }
import os, sys, subprocess, tempfile, urllib.request, logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)   # public dataset - no token needed; silence the rate-limit notice
try:
    import google.colab; IN_COLAB = True
except Exception:
    IN_COLAB = False
import numpy as np, matplotlib.pyplot as plt
try:
    import h5py
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py"], check=False); import h5py

BUNDLE_DIR = ""                          # use a local bundle if present (lab machine)
for c in ("_out/bundle", "../_out/bundle", os.environ.get("ILPEMIS_BUNDLE", ""), "."):
    if c and os.path.exists(os.path.join(c, "table_wiping.h5")):
        BUNDLE_DIR = c; break
if BUNDLE_DIR:
    H5 = os.path.join(BUNDLE_DIR, "table_wiping.h5")
else:                                    # otherwise (Colab): pull the public dataset from Hugging Face
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)
        from huggingface_hub import hf_hub_download
    H5 = hf_hub_download("praneethnamburimit/immersionlab-pe-mis-table-wiping", "table_wiping.h5", repo_type="dataset")

def load_bundle(path):
    def visit(g):
        d = {k: (visit(v) if isinstance(v, h5py.Group) else v[()]) for k, v in g.items()}
        d.update({"@" + k: v for k, v in g.attrs.items()})
        return d
    with h5py.File(path, "r") as h:
        return visit(h)

B = load_bundle(H5)
SEG = {k[1:]: tuple(v) for k, v in B["segments"].items() if k.startswith("@")}
SEG_COLORS = {"Normal": "#3a7d44", "Tensed": "#b03a3a", "Slow": "#999999", "Fast": "#2f6f9f"}
FT = np.asarray(B["us"]["frame_times_ot"])            # OptiTrack time per US frame
P0 = np.asarray(B["us"]["track"]["point0"])           # (n_frames, 2) px
P1 = np.asarray(B["us"]["track"]["point1"])
N = min(len(FT), len(P0), len(P1)); FT, P0, P1 = FT[:N], P0[:N], P1[:N]
US_CLIP_URL = "https://raw.githubusercontent.com/praneethnamburi/immersionlab-pe-mis/main/notebooks/assets/us_tracked_sample.mp4"
print("tracked frames:", N, "| points:", list(B["us"]["track"]), "| IN_COLAB =", IN_COLAB)

In [ ]:
def show_video(path, width=560):
    """Embed a playable clip inline (re-encode to browser-friendly h264 first); fall back to a frame grid."""
    try:
        import cv2
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=False); import cv2
    try:
        h264 = path.rsplit(".", 1)[0] + "_h264.mp4"
        subprocess.run(["ffmpeg", "-y", "-i", path, "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        from IPython.display import Video, display
        display(Video(h264, embed=True, width=width)); return
    except Exception:
        pass
    cap = cv2.VideoCapture(path); frames = []
    while True:
        ok, fr = cap.read()
        if not ok: break
        frames.append(fr)
    cap.release()
    if not frames: return
    idx = np.linspace(0, len(frames) - 1, 6).astype(int)
    fig, ax = plt.subplots(2, 3, figsize=(12, 5))
    for k, ii in enumerate(idx):
        ax.flat[k].imshow(cv2.cvtColor(frames[ii], cv2.COLOR_BGR2RGB)); ax.flat[k].axis("off")
    plt.tight_layout(); plt.show()

## Watch it track — the raw ultrasound

Before any numbers: this is the actual forearm ultrasound during a **Normal** wiping bout (slowed
down), with the two tracked points on it. **Grey** = a near-stable reference; **magenta** = a moving
tissue point; the **green line** is the distance between them. Watch the tissue stretch and relax with
each stroke — that motion is what we're about to measure.

In [ ]:
#@title Play the tracked ultrasound { display-mode: "form" }
US_CLIP = "assets/us_tracked_sample.mp4" if os.path.exists("assets/us_tracked_sample.mp4") \
          else os.path.join(tempfile.gettempdir(), "us_tracked_sample.mp4")
if not os.path.exists(US_CLIP):
    urllib.request.urlretrieve(US_CLIP_URL, US_CLIP)
show_video(US_CLIP)

> **Stop & discuss.** Point a body-pose net (like Session 1's) at this and it fails — there are no
> crisp, unique joints here, just speckle that drifts frame to frame. So how *do* you follow a point on
> tissue? Hold that question; we answer it at the end (that's DUSTrack). For now the tracking is done —
> let's use it.

## The tracked tissue points

Two points were followed in the ultrasound image: `point0` (a near-stable reference) and
`point1` (moving tissue). Below: their image-space paths, and their horizontal position over a
few strokes — `point1` sweeps back and forth with the wiping.

In [ ]:
zoom = (14, 19)  #@param
w = (FT >= zoom[0]) & (FT <= zoom[1])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(P0[:, 0], P0[:, 1], ".", ms=1, color="#888", label="point0 (ref)")
ax[0].plot(P1[:, 0], P1[:, 1], ".", ms=1, color="#7a3fa0", label="point1 (tissue)")
ax[0].invert_yaxis(); ax[0].set_xlabel("x (px)"); ax[0].set_ylabel("y (px)")
ax[0].set_title("image-space paths"); ax[0].legend(); ax[0].set_aspect("equal", "datalim")
ax[1].plot(FT[w], P0[w, 0], color="#888", label="point0 x")
ax[1].plot(FT[w], P1[w, 0], color="#7a3fa0", label="point1 x")
ax[1].set_xlabel("OptiTrack time (s)"); ax[1].set_ylabel("x position (px)")
ax[1].set_title(f"horizontal position, {zoom[0]}-{zoom[1]} s"); ax[1].legend()
plt.tight_layout(); plt.show()

## From tracking to a metric: inter-point distance

A single number per frame that captures **tissue deformation**: the distance between the two
points. It stretches and relaxes with each stroke.

In [ ]:
dist = np.hypot(P1[:, 0] - P0[:, 0], P1[:, 1] - P0[:, 1])
fig, ax = plt.subplots(2, 1, figsize=(13, 6))
ax[0].plot(FT, dist, color="#7a3fa0", lw=0.5)
for k, (a, b) in SEG.items():
    ax[0].axvspan(a, b, color=SEG_COLORS[k], alpha=0.1)
ax[0].set_ylabel("distance (px)"); ax[0].set_title("inter-point distance — whole session")
w = (FT >= zoom[0]) & (FT <= zoom[1])
ax[1].plot(FT[w], dist[w], color="#7a3fa0")
ax[1].set_xlabel("OptiTrack time (s)"); ax[1].set_ylabel("distance (px)")
ax[1].set_title(f"zoom {zoom[0]}-{zoom[1]} s")
for a in ax: a.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## Does the tissue follow the muscle?

Overlay the deformation on the flexor EMG for the same window — the tissue distance changes
right after the muscle fires. And per condition, the deformation *amplitude* (how much the
distance varies) is largest when bracing (Tensed) and during fast strokes.

In [ ]:
flx = B["emg"]["RForearmFlexors"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
w = (FT >= zoom[0]) & (FT <= zoom[1])
we = (flx["t_ot"] >= zoom[0]) & (flx["t_ot"] <= zoom[1])
ax[0].plot(FT[w], dist[w], color="#7a3fa0", label="US distance")
ax0b = ax[0].twinx(); ax0b.plot(flx["t_ot"][we], flx["rms_mV"][we], color="#b03a3a", lw=0.8, label="flexor EMG")
ax[0].set_xlabel("OptiTrack time (s)"); ax[0].set_ylabel("distance (px)", color="#7a3fa0")
ax0b.set_ylabel("flexor EMG (mV)", color="#b03a3a"); ax[0].set_title("deformation vs muscle (zoom)")
conds = [c for c in ("Normal", "Tensed", "Fast") if c in SEG]
defo = [float(np.nanstd(dist[(FT >= SEG[c][0]) & (FT <= SEG[c][1])])) for c in conds]
ax[1].bar(range(len(conds)), defo, color=[SEG_COLORS[c] for c in conds])
ax[1].set_xticks(range(len(conds))); ax[1].set_xticklabels(conds)
ax[1].set_ylabel("deformation (std of distance, px)"); ax[1].set_title("deformation by condition")
plt.tight_layout(); plt.show()
print("deformation (px):", {c: round(v, 2) for c, v in zip(conds, defo)})

## A signature of *how* you moved

Look at the right-hand bars: the tissue deformation isn't the same across conditions — bracing
(**Tensed**) and **Fast** strokes deform the tissue more than **Normal**. So a single number read
from the ultrasound tells you something about *how* the movement was done, not just that it happened.

That's the thread of this whole session. In our research, a related read-out — *how efficiently the
elastic tissue moves* — tracks **general motor skill**: more skilled movers show cleaner, less wasteful
tissue motion ([Namburi et al. 2025](https://doi.org/10.1038/s41598-025-17092-0)). You just found a
smaller version of the same idea in your own data: **signatures of how you move live in soft-tissue motion.**

> **Stop & discuss.** What would you expect *your* tissue signature to look like — relaxed vs. braced?
> fresh vs. fatigued? a skill you're expert at vs. one you're new to?

## Design your own metric

The tracked points are just numbers — you can define whatever measurement answers *your*
question. Pick one below (or ask the assistant to add your own), and see how it behaves.

In [ ]:
metric = "inter-point distance"  #@param ["inter-point distance", "point1 vertical (y)", "point1 frame-to-frame speed"]
if metric == "inter-point distance":
    m = np.hypot(P1[:, 0] - P0[:, 0], P1[:, 1] - P0[:, 1]); unit = "px"
elif metric == "point1 vertical (y)":
    m = P1[:, 1]; unit = "px"
else:
    m = np.r_[0.0, np.hypot(np.diff(P1[:, 0]), np.diff(P1[:, 1]))]; unit = "px/frame"
plt.figure(figsize=(13, 3.5))
plt.plot(FT, m, color="#7a3fa0", lw=0.5)
for k, (a, b) in SEG.items(): plt.axvspan(a, b, color=SEG_COLORS[k], alpha=0.1)
plt.xlabel("OptiTrack time (s)"); plt.ylabel(f"{metric} ({unit})"); plt.title(metric)
plt.tight_layout(); plt.show()

## Your turn

`P0`, `P1` (tracked points), `FT` (their OptiTrack times), the EMG (`B['emg']`), mocap
(`B['mocap']`), and `SEG` are all in scope. Try, or ask the assistant:
- a **lag** between the flexor-EMG burst and the deformation peak (the electromechanical delay);
- an **angle** or area from the two points instead of a distance;
- compare your metric across **Normal / Tensed / Fast** — your own signature.

Next you'll see **how** DUSTrack turns speckle into these tracks — DeepLabCut for the features,
an LK filter for temporal continuity: the markerless idea from Session 1, adapted for a much harder image.